# Lab 2.1 - Build a 3-Layer Neural Network From Scratch

**Dataset:** UCI Occupancy Detection Data Set (real office-room sensor logs)

**Architecture:** 5 inputs -> 8 (ReLU) -> 4 (ReLU) -> 1 (Sigmoid)

Before starting, open `dnn_lib.py` in this folder and complete **TODO 1 - TODO 8**.
Come back to this notebook once `dnn_lib.py` is finished - the cells below import
and use those functions directly, so you'll see errors until they're implemented.

This notebook has its own TODOs too (TODO 9 - TODO 12) for wiring everything
together: normalizing the data, initializing parameters, running the training
loop, and evaluating the trained model.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from dnn_lib import (
    data_normalize, full_forward_propagation, get_cost_value,
    single_layer_backward_propagation, compute_metrics
)

np.random.seed(10)

## Step 1 - Load and inspect the data

We use the three files exactly as provided by the dataset authors:
- `datatraining.txt` - training set
- `datatest.txt` - validation/test set
- `datatest2.txt` - second, independent test set

Each row has a `date` column (not a feature) and 5 sensor features, plus the
`Occupancy` label. This loading step is provided for you.

In [ ]:
import pandas as pd

train_df = pd.read_csv("data/datatraining.txt")
test_df  = pd.read_csv("data/datatest.txt")
test2_df = pd.read_csv("data/datatest2.txt")

FEATURES = ["Temperature", "Humidity", "Light", "CO2", "HumidityRatio"]
LABEL = "Occupancy"

raw_x = train_df[FEATURES].to_numpy(dtype=float)
raw_y = train_df[[LABEL]].to_numpy(dtype=float)

test_x_raw  = test_df[FEATURES].to_numpy(dtype=float)
test_y_raw  = test_df[[LABEL]].to_numpy(dtype=float)

test2_x_raw = test2_df[FEATURES].to_numpy(dtype=float)
test2_y_raw = test2_df[[LABEL]].to_numpy(dtype=float)

print("train:", raw_x.shape, raw_y.shape, "  occupied fraction:", raw_y.mean().round(3))
print("test :", test_x_raw.shape, test_y_raw.shape, "  occupied fraction:", test_y_raw.mean().round(3))
print("test2:", test2_x_raw.shape, test2_y_raw.shape, "  occupied fraction:", test2_y_raw.mean().round(3))

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 7))
for ax, feat in zip(axes.flat, FEATURES):
    ax.hist(train_df[feat], bins=40)
    ax.set_title(feat)
axes.flat[-1].bar(["Not occupied", "Occupied"],
                   [ (raw_y==0).sum(), (raw_y==1).sum() ])
axes.flat[-1].set_title("Class balance (train)")
plt.tight_layout()
plt.savefig("feature_distributions.png", dpi=110)
plt.show()

# Look at the printed ranges above (or the histograms): which feature has the
# smallest numeric range, and which has the largest? What would happen to the
# weighted sums in layer 1 if you skipped normalization?

## Step 2 - Normalize

**TODO 9:** Normalize `raw_x`, `test_x_raw`, and `test2_x_raw` to the [0, 1]
range using `data_normalize()` from `dnn_lib.py` (which you should have
finished implementing as TODO 1). Each split is normalized independently
using its own min/max.

In [ ]:
# TODO 9: normalize all three feature sets using data_normalize()
train_x, train_max = None, None
test_x, _  = None, None
test2_x, _ = None, None

print("train_x range:", train_x.min(), "-", train_x.max())

## Step 3 - Initialize parameters

**TODO 10:** Initialize weights and biases for the 5 -> 8 -> 4 -> 1
architecture. Weights should be small random values (e.g. `np.random.randn(...) * 0.1`),
biases should start at zero.

In [ ]:
INPUT_SIZE  = 5
HID_LAYER1  = 8
HID_LAYER2  = 4
OUTPUT_SIZE = 1

# TODO 10: initialize W1, b1, W2, b2, W3, b3
W1 = None
b1 = None
W2 = None
b2 = None
W3 = None
b3 = None

params = {"W1": W1, "b1": b1, "W2": W2, "b2": b2, "W3": W3, "b3": b3}

## Steps 4-7 - Forward propagation, cost, backpropagation, parameter update

**TODO 11:** Complete the training loop below:
1. Call `full_forward_propagation(X, params)` to get `A3` and `memory`.
2. Call `get_cost_value(A3, Y)` and append it to `cost_history`.
3. The output-layer gradient (`dZ3`, `dW3`, `db3`, `dA2`) is given for you -
   it's the standard sigmoid + binary-cross-entropy shortcut.
4. Chain `single_layer_backward_propagation()` through layer 2 and layer 1.
5. Update all six parameters using gradient descent:
   `param -= learning_rate * d_param`.

In [ ]:
learning_rate = 0.5
num_iterations = 3000

X = train_x.T          # shape (5, m)
Y = raw_y.T            # shape (1, m)

cost_history = []

for i in range(num_iterations):
    # TODO 11a: forward pass
    A3, memory = None, None
    A1, Z1, A2, Z2, Z3 = memory["A1"], memory["Z1"], memory["A2"], memory["Z2"], memory["Z3"]

    # TODO 11b: compute and record the cost
    cost = None
    cost_history.append(cost)

    # Standard simplification for sigmoid + binary cross-entropy: dZ3 = A3 - Y
    m = A2.shape[1]
    dZ3 = A3 - Y
    dW3 = np.dot(dZ3, A2.T) / m
    db3 = np.sum(dZ3, axis=1, keepdims=True) / m
    dA2 = np.dot(params["W3"].T, dZ3)

    # TODO 11c: backprop through layer 2 and layer 1
    dA1, dW2, db2 = None, None, None
    dA0, dW1, db1 = None, None, None

    # TODO 11d: update all six parameters
    pass

    if i % 300 == 0 or i == num_iterations - 1:
        print(f"iteration {i:5d}  cost = {cost:.4f}")

In [ ]:
plt.figure(figsize=(6,4))
plt.plot(cost_history)
plt.xlabel("Iteration")
plt.ylabel("Cost (binary cross-entropy)")
plt.title("Training cost over iterations")
plt.tight_layout()
plt.savefig("cost_history.png", dpi=110)
plt.show()

## Step 8 - Evaluate on `datatest.txt` and `datatest2.txt`

Because ~21% of rows are "Occupied", a model that always predicts "not
occupied" would already score ~79% plain accuracy while being useless.
That's why we also report **precision, recall, and F1**, plus the full
confusion matrix, for both test files.

**TODO 12:** Complete the `evaluate()` function:
1. Run a forward pass on `x_norm.T` using the trained `params`.
2. Threshold the output at 0.5 to get binary predictions.
3. Call `compute_metrics()` to get accuracy/precision/recall/F1/confusion matrix.

In [ ]:
def evaluate(x_norm, y_true, params, name):
    # TODO 12: forward pass, threshold at 0.5, compute metrics
    A3, _ = None, None
    y_pred = None
    metrics = None

    cm = metrics["confusion_matrix"]
    print(f"--- {name} ---")
    print(f"accuracy : {metrics['accuracy']:.4f}")
    print(f"precision: {metrics['precision']:.4f}")
    print(f"recall   : {metrics['recall']:.4f}")
    print(f"f1       : {metrics['f1']:.4f}")
    print(f"confusion matrix -> TP={cm['tp']}  FP={cm['fp']}  TN={cm['tn']}  FN={cm['fn']}")
    print()
    return metrics

metrics_test  = evaluate(test_x,  test_y_raw,  params, "datatest.txt")
metrics_test2 = evaluate(test2_x, test2_y_raw, params, "datatest2.txt")

## Discussion - class imbalance (deliverable)

In one paragraph: compare the plain accuracy above to precision/recall/F1.
Does accuracy alone make the model look better than it is? What does recall
on the "Occupied" class tell you that accuracy hides? How did you notice the
imbalance in the first place - from the histogram in Step 1, or from the
metrics here?